# HD-sEMG Analysis Notebook

**Generated by hdsemg-pipe**

This notebook provides a starting point for analyzing the cleaned motor unit data from your processing pipeline.

## Contents
1. Data Loading
2. Protocol & Condition Grouping
3. Motor Unit Visualizations
4. Motor Unit Analysis
5. Condition Comparison (CON vs EXZ)
   - 5.1 MU Tracking across Conditions
   - 5.2 Unpaired MU Property Overview (+ Plateau Selection)
   - 5.3 Paired Statistical Analysis (+ Washout Check)
6. PIC Analysis (Pyramids - Delta F Method)
7. Custom Analysis (template)
8. Export Results

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import json
import re
import os

# Statistical tests (for paired analysis)
from scipy import stats

# Import helper module (generated alongside this notebook)
import sys
WORKFOLDER = Path.cwd()
if str(WORKFOLDER) not in sys.path:
    sys.path.insert(0, str(WORKFOLDER))

from workfolder_analysis_helper import WorkfolderPaths, MetadataReader, PipelineSummary, ProtocolParser
from workfolder_analysis_helper import plot_pipeline_overview, plot_covisi_comparison
from workfolder_analysis_helper import select_plateau_interactive, recalculate_dr_plateau, apply_plateau_to_all_trapezoids
from workfolder_analysis_helper import create_analysis_folders, save_plot_to_analysis, export_consolidated_csv

# openhdemg (optional but recommended)
try:
    from openhdemg.library import plotemg as plot
    from openhdemg.library import openfiles as emg
    from openhdemg.library import muap
    from openhdemg.library import tools as tools
    OPENHDEMG_AVAILABLE = True
    print("✓ openhdemg available")
except ImportError:
    OPENHDEMG_AVAILABLE = False
    print("⚠ openhdemg not available - install with: pip install openhdemg")

# Matplotlib settings
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# Initialize workfolder paths
paths = WorkfolderPaths(WORKFOLDER)
reader = MetadataReader(paths)
summary = PipelineSummary(paths, reader)

# Create organized analysis folder structure
analysis_folders = create_analysis_folders(WORKFOLDER)
print(f"✓ Created analysis folder structure: {len(analysis_folders)} subfolders")

# List available files
final_files = paths.list_final_json_files()
print(f"Found {len(final_files)} cleaned JSON file(s):")
for f in final_files:
    print(f"  - {f.name}")

In [ ]:
# Display pipeline processing summary
summary.print_full_summary()
print("\n")
summary.print_quality_metrics()

## 1. Load Motor Unit Data

Load all cleaned JSON files using openhdemg.

In [ ]:
# Load all cleaned decomposition files
emgfiles = []

if OPENHDEMG_AVAILABLE:
    for json_path in final_files:
        try:
            emgfile = emg.emg_from_json(str(json_path))
            emgfile = tools.sort_mus(emgfile)
            emgfiles.append({
                'filename': json_path.name,
                'path': json_path,
                'data': emgfile
            })
            print(f"✓ Loaded {json_path.name}: {emgfile['NUMBER_OF_MUS']} MUs, "
                  f"{emgfile['FSAMP']} Hz, "
                  f"{len(emgfile['RAW_SIGNAL'])} samples")
        except Exception as e:
            print(f"✗ Failed to load {json_path.name}: {e}")
else:
    print("Cannot load files - openhdemg not available")

print(f"\nTotal loaded: {len(emgfiles)} file(s)")

## 2. Protocol & Condition Grouping

Parse the hdsemg-trail protocol file to map EMG file groups to experimental conditions
(Baseline, Pre-Intervention, Post-CON, Post-EXZ, etc.) based on the randomization order.

This creates a **grouping of groupings**: `create_emgfile_groups` groups files by
(basename, muscle), then the protocol parser maps those groups to the study conditions.

In [ ]:
# Parse hdsemg-trail protocol
protocol_parser = ProtocolParser(paths)
protocol = protocol_parser.parse()

if protocol:
    meta = protocol['metadata']
    print(f"Protocol file: {protocol['file']}")
    print(f"  PID: {meta.get('pid', 'N/A')}")
    print(f"  Session: {meta.get('session_type', 'N/A')}")
    print(f"  Randomization: {meta.get('randomization', 'N/A')}")
    print(f"  Steps parsed: {len(protocol['steps'])}")

    # Show training block details
    for step in protocol['steps']:
        if step['step_id'].startswith('training_block'):
            mode = step['inputs'].get('Kontraktionsmodus', 'N/A')
            sets = step['inputs'].get('Sätze abgeschlossen', '?')
            borg = step['inputs'].get('Borg CR10 Anstrengung', '?')
            print(f"\n  {step['name']} ({step['step_id']}):")
            print(f"    Mode: {mode}")
            print(f"    Sets: {sets}, Borg: {borg}")
            if step['notes']:
                print(f"    Notes: {step['notes']}")
else:
    print("No protocol file found - condition grouping will not be available")

In [ ]:
# Create EMG file groups and map to experimental conditions
if OPENHDEMG_AVAILABLE and len(emgfiles) > 0 and protocol:
    from hdsemg_pipe.actions.decomposition_export import create_emgfile_groups

    # Step 1: Group files by (basename, muscle)
    groups_result = create_emgfile_groups(emgfiles, strategy='file_and_muscle', concatenate=True)
    print(f"File grouping: {groups_result['summary']['group_count']} groups, "
          f"{groups_result['summary']['ungrouped_files']} ungrouped\n")

    # Step 2: Map groups to experimental conditions via protocol
    condition_result = protocol_parser.create_condition_groups(groups_result)
    protocol_parser.print_condition_summary(condition_result)
else:
    condition_result = None
    if not protocol:
        print("Skipped: no protocol file available")
    else:
        print("Skipped: no EMG files loaded")

In [ ]:
# Access condition data for analysis
# Example: iterate over conditions and tracking types

if condition_result:
    for cond_name, cond_data in condition_result['conditions'].items():
        training_mode = cond_data.get('training_mode')
        block = cond_data.get('block')

        for tracking_type, groups in cond_data['tracking_types'].items():
            for group in groups:
                emgfile = group.get('concatenated')
                if emgfile is None:
                    continue

                n_mus = emgfile['NUMBER_OF_MUS']
                print(f"{cond_name} | {tracking_type} | {group['muscle']}: "
                      f"{n_mus} MUs ({group['count']} file(s))")

    # Example: compare Post-CON vs Post-EXZ Trapezoid groups
    # post_con_trap = condition_result['conditions'].get('Post_CON', {}).get('tracking_types', {}).get('Trapezoid', [])
    # post_exz_trap = condition_result['conditions'].get('Post_EXZ', {}).get('tracking_types', {}).get('Trapezoid', [])

## 3. Motor Unit Visualizations

### Visualizations:
1. **Discharge Times (plot_mupulses)** - Rasterplot showing MU firing patterns
2. **Firing Rate (plot_idr)** - Instantaneous discharge rate over time
3. **Motor Unit Action Potentials (plot_muaps)** - MUAP morphology comparison
4. **Reference Signal (plot_refsig)** - Force/activation context

In [ ]:
# Visualization 1: Discharge Times (Rasterplot)
if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    # Plot first file as example
    emgfile = emgfiles[0]['data']
    filename = emgfiles[0]['filename']

    fig = plot.plot_mupulses(
        emgfile=emgfile,
        linewidths=0.8,
    )
    plt.show()
else:
    print("Cannot plot - no data loaded")

In [ ]:
# Visualization 2: Instantaneous Discharge Rate
if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    emgfile = emgfiles[0]['data']
    filename = emgfiles[0]['filename']

    fig = plot.plot_idr(
        emgfile=emgfile,
        munumber="all",
    )
    plt.show()

In [ ]:
# Visualization 3: Motor Unit Action Potentials
if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    emgfile = emgfiles[0]['data']
    filename = emgfiles[0]['filename']

    # Plot MUAPs for all motor units (overlay)
    try:
        fig = plot.plot_muaps(
            emgfile=emgfile,
            munumber="all",
            channel=0,  # First channel
        )
        plt.show()
    except Exception as e:
        print(f"Note: MUAP plotting requires spike-triggered averaged data: {e}")

In [ ]:
# Visualization 4: Reference Signal (Force/Activation)
if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    emgfile = emgfiles[0]['data']
    filename = emgfiles[0]['filename']

    fig = plot.plot_refsig(
        emgfile=emgfile,
        ylabel="Force",
    )
    plt.show()

## 4. Motor Unit Analysis

### Analysis Focus Areas:
1. **Recruitment Analysis** - Recruitment thresholds and timing
2. **Firing Rate Statistics** - Mean/peak rates, CoVISI verification
3. **Motor Unit Properties** - MU characteristics summary
4. **Quality Metrics** - SIL values, CoVISI comparison

In [ ]:
# Analysis 1: Recruitment Analysis
print("=" * 60)
print("Recruitment Analysis")
print("=" * 60)

if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    for emg_entry in emgfiles:
        emgfile = emg_entry['data']
        filename = emg_entry['filename']

        print(f"\n{filename}:")
        print(f"  Number of Motor Units: {emgfile['NUMBER_OF_MUS']}")

        # Extract recruitment times (first spike of each MU)
        recruitment_times = []
        for mu_idx in range(emgfile['NUMBER_OF_MUS']):
            mupulses = emgfile['MUPULSES'][mu_idx]
            if len(mupulses) > 0:
                first_spike_sample = mupulses[0]
                first_spike_time = first_spike_sample / emgfile['FSAMP']
                recruitment_times.append((mu_idx, first_spike_time))

        if recruitment_times:
            # Sort by recruitment time
            recruitment_times.sort(key=lambda x: x[1])
            print(f"  Recruitment Order (by time):")
            for rank, (mu_idx, rec_time) in enumerate(recruitment_times, 1):
                print(f"    Rank {rank}: MU {mu_idx} at {rec_time:.2f}s")

In [ ]:
# Analysis 2: Firing Rate Statistics & CoVISI Verification
print("=" * 60)
print("Firing Rate Statistics")
print("=" * 60)

if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    for emg_entry in emgfiles:
        emgfile = emg_entry['data']
        filename = emg_entry['filename']

        print(f"\n{filename}:")

        # Compute firing rate statistics for each MU
        for mu_idx in range(emgfile['NUMBER_OF_MUS']):
            mupulses = emgfile['MUPULSES'][mu_idx]
            if len(mupulses) > 1:
                # Calculate inter-spike intervals (ISI)
                isi = np.diff(mupulses) / emgfile['FSAMP']  # in seconds
                instantaneous_rates = 1.0 / isi  # Hz

                mean_rate = np.mean(instantaneous_rates)
                peak_rate = np.max(instantaneous_rates)

                print(f"  MU {mu_idx}: Mean Rate = {mean_rate:.2f} Hz, Peak Rate = {peak_rate:.2f} Hz")

# Display CoVISI post-validation results
print("\n" + "=" * 60)
print("CoVISI Post-Validation")
print("=" * 60)
post_covisi = reader.get_covisi_post_validation()
if post_covisi:
    print("✓ CoVISI validation performed")
    # You can extract and display specific metrics from the report here
else:
    print("⚠ CoVISI post-validation not performed")

In [ ]:
# Analysis 3: Motor Unit Properties Summary
print("=" * 60)
print("Motor Unit Properties Summary")
print("=" * 60)

properties_list = []

if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    for emg_entry in emgfiles:
        emgfile = emg_entry['data']
        filename = emg_entry['filename']

        for mu_idx in range(emgfile['NUMBER_OF_MUS']):
            mupulses = emgfile['MUPULSES'][mu_idx]

            if len(mupulses) > 1:
                # Calculate properties
                n_spikes = len(mupulses)
                duration = (mupulses[-1] - mupulses[0]) / emgfile['FSAMP']
                isi = np.diff(mupulses) / emgfile['FSAMP']
                mean_firing_rate = np.mean(1.0 / isi) if len(isi) > 0 else 0

                properties_list.append({
                    'File': filename,
                    'MU_Index': mu_idx,
                    'N_Spikes': n_spikes,
                    'Duration_s': duration,
                    'Mean_Firing_Rate_Hz': mean_firing_rate
                })

    # Create DataFrame for easy analysis
    df_properties = pd.DataFrame(properties_list)
    print(df_properties.describe())
    print(f"\n✓ Motor unit properties extracted: {len(df_properties)} MUs")

In [ ]:
# Analysis 4: Quality Metrics Overview
print("=" * 60)
print("Quality Metrics Overview")
print("=" * 60)

# SIL values from decomposition
if OPENHDEMG_AVAILABLE and len(emgfiles) > 0:
    print("\nSilhouette (SIL) Values:")
    for emg_entry in emgfiles:
        emgfile = emg_entry['data']
        filename = emg_entry['filename']

        if 'ACCURACY' in emgfile and emgfile['ACCURACY'] is not None:
            sil_values = emgfile['ACCURACY'].values.flatten()
            print(f"  {filename}:")
            print(f"    Mean SIL: {np.mean(sil_values):.3f}")
            print(f"    Min SIL: {np.min(sil_values):.3f}")
            print(f"    Max SIL: {np.max(sil_values):.3f}")

# CoVISI comparison
print("\n" + "=" * 60)
print("CoVISI Pre/Post Comparison")
print("=" * 60)

pre_report = reader.get_covisi_pre_filter()
post_report = reader.get_covisi_post_validation()

if pre_report:
    print(f"Pre-Filter:")
    removed = pre_report.get('removed_mu_indices', [])
    print(f"  Motor units removed: {len(removed)}")
    print(f"  Threshold: {pre_report.get('threshold', 30.0)}%")

if post_report:
    print("Post-Validation: Quality verified after MUedit cleaning")

## 5. Condition Comparison: CON vs EXZ

Analyse der Effekte akuter konzentrischer und exzentrischer Trainingsinterventionen
auf Motor Unit Eigenschaften.

### Analysierte MU-Parameter:
| Parameter | Beschreibung |
|---|---|
| **MU Yield** | Anzahl identifizierter Motor Units |
| **Mean Discharge Rate (DR)** | Mittlere Entladungsrate (pps) |
| **Peak Discharge Rate** | Maximale Entladungsrate (pps) |
| **Recruitment Threshold (RT)** | Kraft bei Erst-Rekrutierung (%REF) |
| **Derecruitment Threshold (DRT)** | Kraft bei Derekrutierung (%REF) |
| **DR at Recruitment** | Entladungsrate bei Rekrutierung (pps) |
| **DR at Derecruitment** | Entladungsrate bei Derekrutierung (pps) |
| **CoV ISI** | Variationskoeffizient der Interspike-Intervalle (%) |
| **SIL (Accuracy)** | Silhouette-Wert (Dekompositionsqualität) |

### 5.1 MU Tracking across Conditions

Track **identical motor units** from Pre-Intervention to Post-CON and Post-EXZ using
2D cross-correlation of Spike-Triggered Averages (MUAP morphology matching via `muap.tracking()`).

This enables **paired statistical analysis** with significantly higher statistical power
compared to unpaired tests.

**Workflow:**
1. Match individual EMG files between Pre and Post conditions by electrode grid
2. Compute STAs and run 2D cross-correlation to identify the same MUs
3. Extract paired MU properties (Pre vs Post) for tracked units
4. Paired t-tests / Wilcoxon signed-rank tests with effect sizes

In [ ]:
# ============================================================
# MU Tracking Configuration
# ============================================================
# Adjust these parameters for your electrode setup.
# The GUI allows visual inspection of each matched MU pair.

TRACKING_CONFIG = {
    'matrixcode': "GR08MM1305",   # electrode array (GR08MM1305, GR04MM1305, GR10MM0808)
    'orientation': 180,            # electrode orientation in degrees (0 or 180)
    'derivation': "sd",            # single differential (sd), double differential (dd), or monopolar (mono)
    'timewindow': 50,              # ms for STA computation
    'threshold': 0.8,              # minimum normalized 2D XCC for match (0.75=lenient, 0.9=strict)
    'filter': True,                # keep only the best match per MU
    'exclude_belowthreshold': True,
    'gui': True,                   # visual inspection of matches (set False for batch processing)
    'show': False,
    'multiprocessing': True,
}

# Comparisons to run: (name, pre_condition, post_condition)
TRACKING_COMPARISONS = [
    ('Pre_vs_PostCON', 'Pre_Intervention', 'Post_CON'),
    ('Pre_vs_PostEXZ', 'Pre_Intervention', 'Post_EXZ'),
]

print("Tracking configuration:")
for k, v in TRACKING_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# ============================================================
# Run MU Tracking: Match files by grid, track MUs via MUAP morphology
# ============================================================

def get_grid_id(filename):
    """Extract grid identifier (e.g., '10mm_4x8_2') from filename."""
    match = re.search(r'(\d+mm_\d+x\d+(?:_\d+)?)', os.path.basename(filename))
    return match.group(1) if match else 'unknown'

# Build lookup: condition -> tracking_type -> muscle -> grid_id -> file_entry
file_lookup = {}
if condition_result:
    for cond_name, cond_data in condition_result['conditions'].items():
        file_lookup[cond_name] = {}
        for tracking_type, groups in cond_data['tracking_types'].items():
            file_lookup[cond_name][tracking_type] = {}
            for group in groups:
                muscle = group.get('muscle', 'Unknown')
                if muscle not in file_lookup[cond_name][tracking_type]:
                    file_lookup[cond_name][tracking_type][muscle] = {}
                for file_entry in group.get('files', []):
                    grid_id = get_grid_id(file_entry['filename'])
                    file_lookup[cond_name][tracking_type][muscle][grid_id] = file_entry

# Run tracking for each comparison
tracking_results = []

if condition_result and OPENHDEMG_AVAILABLE:
    for comp_name, pre_cond, post_cond in TRACKING_COMPARISONS:
        pre_data = file_lookup.get(pre_cond, {})
        post_data = file_lookup.get(post_cond, {})

        for tracking_type in sorted(pre_data.keys()):
            if tracking_type not in post_data:
                print(f"Skipping {tracking_type}: not in {post_cond}")
                continue

            for muscle in sorted(pre_data[tracking_type].keys()):
                if muscle not in post_data.get(tracking_type, {}):
                    print(f"Skipping {muscle}: not in {post_cond}/{tracking_type}")
                    continue

                pre_files = pre_data[tracking_type][muscle]
                post_files = post_data[tracking_type][muscle]
                common_grids = sorted(set(pre_files.keys()) & set(post_files.keys()))

                if not common_grids:
                    print(f"No matching grids for {comp_name} | {tracking_type} | {muscle}")
                    continue

                for grid_id in common_grids:
                    pre_entry = pre_files[grid_id]
                    post_entry = post_files[grid_id]
                    pre_emg = pre_entry['data']
                    post_emg = post_entry['data']

                    print(f"\nTracking: {comp_name} | {tracking_type} | {muscle} | Grid: {grid_id}")
                    print(f"  Pre:  {os.path.basename(pre_entry['filename'])} "
                          f"({pre_emg['NUMBER_OF_MUS']} MUs)")
                    print(f"  Post: {os.path.basename(post_entry['filename'])} "
                          f"({post_emg['NUMBER_OF_MUS']} MUs)")

                    try:
                        track_res = muap.tracking(
                            emgfile1=pre_emg,
                            emgfile2=post_emg,
                            **TRACKING_CONFIG,
                        )

                        # Filter to included matches (GUI adds 'Inclusion' column)
                        if 'Inclusion' in track_res.columns:
                            matched = track_res[track_res['Inclusion'] == 'Included'].copy()
                        else:
                            matched = track_res.copy()

                        tracking_results.append({
                            'comparison': comp_name,
                            'tracking_type': tracking_type,
                            'muscle': muscle,
                            'grid_id': grid_id,
                            'pre_file': pre_entry['filename'],
                            'post_file': post_entry['filename'],
                            'pre_emgfile': pre_emg,
                            'post_emgfile': post_emg,
                            'tracking_df': track_res,
                            'matched': matched,
                            'n_matched': len(matched),
                            'n_pre_mus': pre_emg['NUMBER_OF_MUS'],
                            'n_post_mus': post_emg['NUMBER_OF_MUS'],
                        })

                        if len(matched) > 0:
                            print(f"  -> {len(matched)} matched pairs "
                                  f"(mean XCC: {matched['XCC'].mean():.3f})")
                        else:
                            print(f"  -> No matches above threshold {TRACKING_CONFIG['threshold']}")

                    except Exception as e:
                        print(f"  ERROR: {e}")

    print(f"\n{'='*60}")
    print(f"Tracking complete:")
    print(f"  Runs: {len(tracking_results)}")
    print(f"  Total matched MU pairs: {sum(r['n_matched'] for r in tracking_results)}")
else:
    print("Tracking requires condition_result and openhdemg")

In [ ]:
# ============================================================
# Extract paired MU properties for tracked motor units
# ============================================================

def extract_mu_properties(emgfile, mu_idx, fsamp, ref_signal, ref_max, sil_values):
    """Extract properties for a single MU."""
    mupulses = emgfile['MUPULSES'][mu_idx]
    if len(mupulses) < 4:
        return None

    spikes = np.array(mupulses)
    isi = np.diff(spikes) / fsamp  # seconds
    isi = isi[isi > 0.01]  # remove artifacts < 10ms
    if len(isi) < 2:
        return None
    inst_dr = 1.0 / isi  # pps

    first_spike, last_spike = spikes[0], spikes[-1]
    rt = (ref_signal[first_spike] / ref_max) * 100 if first_spike < len(ref_signal) else np.nan
    drt = (ref_signal[last_spike] / ref_max) * 100 if last_spike < len(ref_signal) else np.nan

    return {
        'n_spikes': len(spikes),
        'mean_dr': np.mean(inst_dr),
        'peak_dr': np.max(inst_dr),
        'cov_isi': (np.std(isi) / np.mean(isi)) * 100,
        'rt_pct': rt,
        'drt_pct': drt,
        'dr_at_rec': np.mean(inst_dr[:min(5, len(inst_dr))]),
        'dr_at_derec': np.mean(inst_dr[max(0, len(inst_dr)-5):]),
        'sil': sil_values[mu_idx] if sil_values is not None and mu_idx < len(sil_values) else np.nan,
    }


paired_records = []

for result in tracking_results:
    pre_emg = result['pre_emgfile']
    post_emg = result['post_emgfile']
    fsamp = pre_emg['FSAMP']

    pre_ref = pre_emg['REF_SIGNAL'].values.flatten()
    post_ref = post_emg['REF_SIGNAL'].values.flatten()
    pre_ref_max = pre_ref.max() if pre_ref.max() > 0 else 1.0
    post_ref_max = post_ref.max() if post_ref.max() > 0 else 1.0

    pre_sil = (pre_emg['ACCURACY'].values.flatten()
               if 'ACCURACY' in pre_emg and pre_emg['ACCURACY'] is not None else None)
    post_sil = (post_emg['ACCURACY'].values.flatten()
                if 'ACCURACY' in post_emg and post_emg['ACCURACY'] is not None else None)

    for _, row in result['matched'].iterrows():
        mu_pre = int(row['MU_file1'])
        mu_post = int(row['MU_file2'])
        xcc = row['XCC']

        pre_props = extract_mu_properties(pre_emg, mu_pre, fsamp, pre_ref, pre_ref_max, pre_sil)
        post_props = extract_mu_properties(post_emg, mu_post, fsamp, post_ref, post_ref_max, post_sil)

        if pre_props is None or post_props is None:
            continue

        record = {
            'comparison': result['comparison'],
            'tracking_type': result['tracking_type'],
            'muscle': result['muscle'],
            'grid_id': result['grid_id'],
            'mu_pre_idx': mu_pre,
            'mu_post_idx': mu_post,
            'xcc': xcc,
        }
        for key in pre_props:
            record[f'pre_{key}'] = pre_props[key]
            record[f'post_{key}'] = post_props[key]
            record[f'delta_{key}'] = post_props[key] - pre_props[key]

        paired_records.append(record)

df_paired = pd.DataFrame(paired_records)

if len(df_paired) > 0:
    print(f"Paired MU dataset: {len(df_paired)} tracked motor unit pairs")
    n_con = len(df_paired[df_paired['comparison'] == 'Pre_vs_PostCON'])
    n_exz = len(df_paired[df_paired['comparison'] == 'Pre_vs_PostEXZ'])
    print(f"  Pre -> Post-CON: {n_con} pairs")
    print(f"  Pre -> Post-EXZ: {n_exz} pairs")
    print(f"  Muscles: {df_paired['muscle'].unique().tolist()}")
    print(f"  Mean XCC: {df_paired['xcc'].mean():.3f} "
          f"(range: {df_paired['xcc'].min():.3f} - {df_paired['xcc'].max():.3f})")
    print()
    print(df_paired.groupby(['comparison', 'tracking_type', 'muscle']).agg(
        n_pairs=('xcc', 'count'),
        mean_XCC=('xcc', 'mean'),
        delta_DR=('delta_mean_dr', 'mean'),
        delta_RT=('delta_rt_pct', 'mean'),
    ).round(3))
else:
    print("No paired MU data - tracking found no matches or was skipped")

In [ ]:
# ============================================================
# Tracking Quality: XCC Distribution of Matched MU Pairs
# ============================================================
if len(df_paired) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # XCC histogram
    ax = axes[0]
    for comp, color, label in [('Pre_vs_PostCON', '#2563eb', 'Post-CON'),
                                ('Pre_vs_PostEXZ', '#dc2626', 'Post-EXZ')]:
        subset = df_paired[df_paired['comparison'] == comp]
        if not subset.empty:
            ax.hist(subset['xcc'], bins=15, alpha=0.5, color=color, edgecolor='black',
                    linewidth=0.5, label=f'{label} (n={len(subset)})')
    ax.set_xlabel('Normalized 2D Cross-Correlation (XCC)')
    ax.set_ylabel('Count')
    ax.set_title('Tracking Quality: XCC Distribution')
    ax.axvline(TRACKING_CONFIG['threshold'], color='black', linestyle='--',
               linewidth=1, label=f'Threshold ({TRACKING_CONFIG["threshold"]})')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    # XCC per muscle/tracking type
    ax = axes[1]
    plot_data = df_paired.groupby(['muscle', 'comparison'])['xcc'].apply(list)
    x_pos = 0
    xticks, xlabels = [], []
    for muscle in sorted(df_paired['muscle'].unique()):
        muscle_short = muscle.replace('Right', '').replace('Left', '')
        for comp, color in [('Pre_vs_PostCON', '#2563eb'), ('Pre_vs_PostEXZ', '#dc2626')]:
            vals = df_paired[(df_paired['muscle'] == muscle) &
                             (df_paired['comparison'] == comp)]['xcc'].values
            if len(vals) > 0:
                bp = ax.boxplot([vals], positions=[x_pos], widths=0.6, patch_artist=True,
                               medianprops=dict(color='black'))
                bp['boxes'][0].set_facecolor(color)
                bp['boxes'][0].set_alpha(0.6)
                label = comp.replace('Pre_vs_Post', '')
                xticks.append(x_pos)
                xlabels.append(f'{muscle_short}\n{label}')
                x_pos += 1
        x_pos += 0.5

    ax.set_xticks(xticks)
    ax.set_xticklabels(xlabels, fontsize=8)
    ax.set_ylabel('XCC')
    ax.set_title('XCC per Muscle & Comparison')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    save_path = save_plot_to_analysis(plt.gcf(), 'fig_tracking_quality.png', 'tracking')
    print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

### 5.2 Unpaired MU Property Overview

Descriptive overview of **all** motor unit properties across conditions (not limited to tracked pairs).
This uses the concatenated EMG files from `create_emgfile_groups()`.

In [ ]:
# ============================================================
# Extract MU properties from all conditions into a DataFrame
# ============================================================

CONDITION_ORDER = ['Baseline', 'Pre_Intervention', 'Post_CON', 'Post_EXZ', 'Post_Washout', 'Final']

mu_records = []

if condition_result and OPENHDEMG_AVAILABLE:
    for cond_name, cond_data in condition_result['conditions'].items():
        block = cond_data.get('block', '')
        training_mode = cond_data.get('training_mode')

        for tracking_type, groups in cond_data['tracking_types'].items():
            for group in groups:
                emgfile = group.get('concatenated')
                if emgfile is None:
                    continue

                fsamp = emgfile['FSAMP']
                n_mus = emgfile['NUMBER_OF_MUS']
                ref_signal = emgfile['REF_SIGNAL'].values.flatten()
                ref_max = ref_signal.max() if ref_signal.max() > 0 else 1.0

                # SIL / Accuracy values
                sil_values = None
                if 'ACCURACY' in emgfile and emgfile['ACCURACY'] is not None:
                    sil_values = emgfile['ACCURACY'].values.flatten()

                for mu_idx in range(n_mus):
                    mupulses = emgfile['MUPULSES'][mu_idx]
                    if len(mupulses) < 4:
                        continue

                    spikes = np.array(mupulses)

                    # ISI and firing rates
                    isi = np.diff(spikes) / fsamp  # seconds
                    isi = isi[isi > 0.01]  # remove artifacts < 10ms
                    if len(isi) < 2:
                        continue
                    inst_dr = 1.0 / isi  # pps

                    mean_dr = np.mean(inst_dr)
                    peak_dr = np.max(inst_dr)
                    cov_isi = (np.std(isi) / np.mean(isi)) * 100  # %

                    # Recruitment & derecruitment
                    first_spike = spikes[0]
                    last_spike = spikes[-1]
                    rt = (ref_signal[first_spike] / ref_max) * 100 if first_spike < len(ref_signal) else np.nan
                    drt = (ref_signal[last_spike] / ref_max) * 100 if last_spike < len(ref_signal) else np.nan

                    # DR at recruitment (first 5 ISIs) and derecruitment (last 5 ISIs)
                    dr_at_rec = np.mean(inst_dr[:min(5, len(inst_dr))])
                    dr_at_derec = np.mean(inst_dr[max(0, len(inst_dr)-5):])

                    # SIL
                    sil = sil_values[mu_idx] if sil_values is not None and mu_idx < len(sil_values) else np.nan

                    mu_records.append({
                        'condition': cond_name,
                        'block': block,
                        'training_mode': training_mode if training_mode else 'None',
                        'tracking_type': tracking_type,
                        'muscle': group.get('muscle', 'Unknown'),
                        'group_name': group['name'],
                        'mu_idx': mu_idx,
                        'n_spikes': len(spikes),
                        'mean_dr': mean_dr,
                        'peak_dr': peak_dr,
                        'cov_isi': cov_isi,
                        'rt_pct': rt,
                        'drt_pct': drt,
                        'dr_at_rec': dr_at_rec,
                        'dr_at_derec': dr_at_derec,
                        'sil': sil,
                    })

    df_mu = pd.DataFrame(mu_records)
    # Set condition as ordered categorical for correct plot ordering
    df_mu['condition'] = pd.Categorical(df_mu['condition'], categories=CONDITION_ORDER, ordered=True)
    df_mu = df_mu.sort_values('condition')

    print(f"Extracted {len(df_mu)} motor units across {df_mu['condition'].nunique()} conditions")
    print(f"Muscles: {df_mu['muscle'].unique().tolist()}")
    print(f"Tracking types: {df_mu['tracking_type'].unique().tolist()}")
    print()
    print(df_mu.groupby(['condition', 'tracking_type', 'muscle']).agg(
        n_MUs=('mu_idx', 'count'),
        mean_DR=('mean_dr', 'mean'),
        mean_RT=('rt_pct', 'mean'),
    ).round(2))
else:
    df_mu = pd.DataFrame()
    print("No condition data available")

In [ ]:
# ============================================================
# Figure 1: Motor Unit Yield across Conditions
# ============================================================
if len(df_mu) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, tracking in zip(axes, ['Trapezoid', 'Pyramid']):
        subset = df_mu[df_mu['tracking_type'] == tracking]
        if subset.empty:
            ax.set_title(f'{tracking} (no data)')
            continue

        # Count MUs per condition and muscle
        counts = subset.groupby(['condition', 'muscle']).size().unstack(fill_value=0)

        counts.plot(kind='bar', ax=ax, width=0.7, edgecolor='black', linewidth=0.5)
        ax.set_title(f'{tracking} Tracking', fontsize=13, fontweight='bold')
        ax.set_xlabel('')
        ax.set_ylabel('Number of Motor Units')
        ax.tick_params(axis='x', rotation=35)
        ax.legend(title='Muscle', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

        # Highlight CON/EXZ blocks
        for i, label in enumerate(ax.get_xticklabels()):
            txt = label.get_text()
            if 'Post_CON' in txt:
                label.set_color('#2563eb')
                label.set_fontweight('bold')
            elif 'Post_EXZ' in txt:
                label.set_color('#dc2626')
                label.set_fontweight('bold')

    fig.suptitle('Motor Unit Yield per Condition', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = save_plot_to_analysis(fig, 'fig_mu_yield_conditions.png', 'comparisons')
    print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

In [ ]:
# ============================================================
# Figure 2: Mean Discharge Rate across Conditions (Boxplot)
# ============================================================
if len(df_mu) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, tracking in zip(axes, ['Trapezoid', 'Pyramid']):
        subset = df_mu[df_mu['tracking_type'] == tracking]
        if subset.empty:
            ax.set_title(f'{tracking} (no data)')
            continue

        # Boxplot: one box per condition, colored by muscle
        muscles = subset['muscle'].unique()
        conditions = [c for c in CONDITION_ORDER if c in subset['condition'].values]
        n_conds = len(conditions)
        width = 0.35
        x = np.arange(n_conds)

        for j, muscle in enumerate(muscles):
            muscle_data = subset[subset['muscle'] == muscle]
            bp_data = [muscle_data[muscle_data['condition'] == c]['mean_dr'].dropna().values
                       for c in conditions]
            positions = x + (j - 0.5 * (len(muscles) - 1)) * width
            bp = ax.boxplot(bp_data, positions=positions, widths=width * 0.85,
                           patch_artist=True, showfliers=True, medianprops=dict(color='black'))
            color = '#3b82f6' if j == 0 else '#f97316'
            for patch in bp['boxes']:
                patch.set_facecolor(color)
                patch.set_alpha(0.6)
            # Legend proxy
            ax.plot([], [], 's', color=color, alpha=0.6, markersize=10, label=muscle)

        ax.set_xticks(x)
        ax.set_xticklabels(conditions, rotation=35, ha='right')
        ax.set_title(f'{tracking} Tracking', fontsize=13, fontweight='bold')
        ax.set_ylabel('Mean Discharge Rate (pps)')
        ax.legend(title='Muscle', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Mean Discharge Rate across Conditions', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = save_plot_to_analysis(fig, 'fig_mean_dr_conditions.png', 'comparisons')
    print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

In [ ]:
# ============================================================
# Figure 3: Recruitment Threshold across Conditions (Boxplot)
# ============================================================
if len(df_mu) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    for ax, tracking in zip(axes, ['Trapezoid', 'Pyramid']):
        subset = df_mu[df_mu['tracking_type'] == tracking]
        if subset.empty:
            ax.set_title(f'{tracking} (no data)')
            continue

        muscles = subset['muscle'].unique()
        conditions = [c for c in CONDITION_ORDER if c in subset['condition'].values]
        n_conds = len(conditions)
        width = 0.35
        x = np.arange(n_conds)

        for j, muscle in enumerate(muscles):
            muscle_data = subset[subset['muscle'] == muscle]
            bp_data = [muscle_data[muscle_data['condition'] == c]['rt_pct'].dropna().values
                       for c in conditions]
            positions = x + (j - 0.5 * (len(muscles) - 1)) * width
            bp = ax.boxplot(bp_data, positions=positions, widths=width * 0.85,
                           patch_artist=True, showfliers=True, medianprops=dict(color='black'))
            color = '#3b82f6' if j == 0 else '#f97316'
            for patch in bp['boxes']:
                patch.set_facecolor(color)
                patch.set_alpha(0.6)
            ax.plot([], [], 's', color=color, alpha=0.6, markersize=10, label=muscle)

        ax.set_xticks(x)
        ax.set_xticklabels(conditions, rotation=35, ha='right')
        ax.set_title(f'{tracking} Tracking', fontsize=13, fontweight='bold')
        ax.set_ylabel('Recruitment Threshold (%REF)')
        ax.legend(title='Muscle', fontsize=8)
        ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Recruitment Threshold across Conditions', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = save_plot_to_analysis(fig, 'fig_rt_conditions.png', 'comparisons')
    print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

In [ ]:
# ============================================================
# Figure 4: Timeline Plot - MU Properties across Blocks
# ============================================================
if len(df_mu) > 0:
    metrics = {
        'mean_dr': ('Mean Discharge Rate (pps)', '#2563eb'),
        'peak_dr': ('Peak Discharge Rate (pps)', '#7c3aed'),
        'rt_pct': ('Recruitment Threshold (%REF)', '#059669'),
        'drt_pct': ('Derecruitment Threshold (%REF)', '#d97706'),
    }

    for tracking in ['Trapezoid', 'Pyramid']:
        subset = df_mu[df_mu['tracking_type'] == tracking]
        if subset.empty:
            continue

        fig, axes = plt.subplots(2, 2, figsize=(14, 10))

        for ax, (metric, (label, color)) in zip(axes.flat, metrics.items()):
            for muscle in subset['muscle'].unique():
                m_data = subset[subset['muscle'] == muscle]
                conditions = [c for c in CONDITION_ORDER if c in m_data['condition'].values]

                means = [m_data[m_data['condition'] == c][metric].mean() for c in conditions]
                sems = [m_data[m_data['condition'] == c][metric].sem() for c in conditions]

                style = '-o' if 'Lateralis' in muscle else '-s'
                ax.errorbar(range(len(conditions)), means, yerr=sems, fmt=style,
                           capsize=4, capthick=1.5, linewidth=2, markersize=7,
                           label=muscle, alpha=0.85)

            ax.set_xticks(range(len(conditions)))
            ax.set_xticklabels(conditions, rotation=35, ha='right', fontsize=9)
            ax.set_ylabel(label, fontsize=10)
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)

            # Shade training periods
            for i, c in enumerate(conditions):
                if 'Post_CON' in c:
                    ax.axvspan(i - 0.4, i + 0.4, alpha=0.08, color='blue')
                elif 'Post_EXZ' in c:
                    ax.axvspan(i - 0.4, i + 0.4, alpha=0.08, color='red')

        fig.suptitle(f'{tracking} Tracking - MU Properties Timeline (Mean \u00b1 SEM)',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        save_path = save_plot_to_analysis(fig, f'fig_timeline_{tracking.lower()}.png', 'timelines')
        print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

In [ ]:
# ============================================================
# EXPLORATORY: Recruitment Threshold vs Discharge Rate (Onion Skin)
# ============================================================
# Uncomment to generate this exploratory plot (set to True):
if False and len(df_mu) > 0:
    focus_conditions = ['Pre_Intervention', 'Post_CON', 'Post_EXZ']
    cond_colors = {'Pre_Intervention': '#6b7280', 'Post_CON': '#2563eb', 'Post_EXZ': '#dc2626'}
    cond_markers = {'Pre_Intervention': 'o', 'Post_CON': 's', 'Post_EXZ': '^'}

    for tracking in ['Trapezoid', 'Pyramid']:
        subset = df_mu[(df_mu['tracking_type'] == tracking) &
                       (df_mu['condition'].isin(focus_conditions))]
        if subset.empty:
            continue

        muscles = subset['muscle'].unique()
        fig, axes = plt.subplots(1, len(muscles), figsize=(7 * len(muscles), 6), squeeze=False)

        for ax, muscle in zip(axes[0], muscles):
            m_data = subset[subset['muscle'] == muscle]

            for cond in focus_conditions:
                c_data = m_data[m_data['condition'] == cond]
                if c_data.empty:
                    continue
                ax.scatter(c_data['rt_pct'], c_data['mean_dr'],
                          c=cond_colors[cond], marker=cond_markers[cond],
                          s=60, alpha=0.7, edgecolors='black', linewidth=0.5,
                          label=cond)

            ax.set_xlabel('Recruitment Threshold (%REF)', fontsize=11)
            ax.set_ylabel('Mean Discharge Rate (pps)', fontsize=11)
            ax.set_title(f'{muscle}', fontsize=12, fontweight='bold')
            ax.legend(fontsize=9)
            ax.grid(alpha=0.3)

        fig.suptitle(f'{tracking} - RT vs DR (Onion Skin Pattern)',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        save_path = save_plot_to_analysis(fig, f'fig_rt_vs_dr_{tracking.lower()}.png', 'exploratory')
        print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

In [ ]:
# ============================================================
# Table: Summary Statistics per Condition
# ============================================================
if len(df_mu) > 0:
    summary_cols = ['mean_dr', 'peak_dr', 'rt_pct', 'drt_pct', 'dr_at_rec', 'dr_at_derec', 'cov_isi', 'sil']
    col_labels = {
        'mean_dr': 'Mean DR (pps)',
        'peak_dr': 'Peak DR (pps)',
        'rt_pct': 'RT (%REF)',
        'drt_pct': 'DRT (%REF)',
        'dr_at_rec': 'DR@Rec (pps)',
        'dr_at_derec': 'DR@Derec (pps)',
        'cov_isi': 'CoV ISI (%)',
        'sil': 'SIL',
    }

    for tracking in ['Trapezoid', 'Pyramid']:
        subset = df_mu[df_mu['tracking_type'] == tracking]
        if subset.empty:
            continue

        print(f"\n{'='*80}")
        print(f"  {tracking} Tracking - Summary Statistics (Mean \u00b1 SD)")
        print(f"{'='*80}")

        for muscle in sorted(subset['muscle'].unique()):
            m_data = subset[subset['muscle'] == muscle]
            print(f"\n  {muscle}:")
            print(f"  {'Condition':<20s} {'n':>4s}", end='')
            for col in summary_cols:
                print(f"  {col_labels[col]:>14s}", end='')
            print()
            print(f"  {'-'*20} {'-'*4}", end='')
            for _ in summary_cols:
                print(f"  {'-'*14}", end='')
            print()

            for cond in CONDITION_ORDER:
                c_data = m_data[m_data['condition'] == cond]
                if c_data.empty:
                    continue
                n = len(c_data)
                marker = ' *' if cond in ['Post_CON', 'Post_EXZ'] else '  '
                print(f"{marker}{cond:<20s} {n:>4d}", end='')
                for col in summary_cols:
                    vals = c_data[col].dropna()
                    if len(vals) > 0:
                        print(f"  {vals.mean():>6.1f}\u00b1{vals.std():>5.1f}", end='')
                    else:
                        print(f"  {'N/A':>14s}", end='')
                print()

    # DataFrame will be included in consolidated CSV export later
    print(f"\n✓ MU properties by condition ready: {len(df_mu)} entries")

In [ ]:
# ============================================================
# EXPLORATORY: Direct CON vs EXZ Comparison (Pre -> Post delta)
# ============================================================
# Uncomment to generate this exploratory plot (set to True):
if False and len(df_mu) > 0:
    # Calculate condition means per muscle/tracking for Pre vs Post comparisons
    key_metrics = ['mean_dr', 'peak_dr', 'rt_pct', 'cov_isi']
    metric_labels = {
        'mean_dr': '\u0394 Mean DR (pps)',
        'peak_dr': '\u0394 Peak DR (pps)',
        'rt_pct': '\u0394 RT (%REF)',
        'cov_isi': '\u0394 CoV ISI (%)',
    }

    for tracking in ['Trapezoid', 'Pyramid']:
        subset = df_mu[df_mu['tracking_type'] == tracking]
        if subset.empty:
            continue

        fig, axes = plt.subplots(1, len(key_metrics), figsize=(4 * len(key_metrics), 5))

        for ax, metric in zip(axes, key_metrics):
            bars_data = []
            bar_labels = []
            bar_colors = []

            for muscle in sorted(subset['muscle'].unique()):
                m_data = subset[subset['muscle'] == muscle]
                pre = m_data[m_data['condition'] == 'Pre_Intervention'][metric].dropna()
                post_con = m_data[m_data['condition'] == 'Post_CON'][metric].dropna()
                post_exz = m_data[m_data['condition'] == 'Post_EXZ'][metric].dropna()

                pre_mean = pre.mean() if len(pre) > 0 else np.nan
                muscle_short = muscle.replace('Right', '').replace('Left', '')

                if len(post_con) > 0 and not np.isnan(pre_mean):
                    delta_con = post_con.mean() - pre_mean
                    bars_data.append(delta_con)
                    bar_labels.append(f'{muscle_short}\nCON')
                    bar_colors.append('#2563eb')

                if len(post_exz) > 0 and not np.isnan(pre_mean):
                    delta_exz = post_exz.mean() - pre_mean
                    bars_data.append(delta_exz)
                    bar_labels.append(f'{muscle_short}\nEXZ')
                    bar_colors.append('#dc2626')

            if bars_data:
                x = np.arange(len(bars_data))
                ax.bar(x, bars_data, color=bar_colors, alpha=0.7, edgecolor='black', linewidth=0.5)
                ax.set_xticks(x)
                ax.set_xticklabels(bar_labels, fontsize=8)
                ax.axhline(0, color='black', linewidth=0.8, linestyle='-')
                ax.set_ylabel(metric_labels[metric], fontsize=10)
                ax.grid(axis='y', alpha=0.3)

        fig.suptitle(f'{tracking} - Change from Pre-Intervention (Post - Pre)',
                     fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()
        save_path = save_plot_to_analysis(fig, f'fig_delta_con_exz_{tracking.lower()}.png', 'exploratory')
        print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

#### 5.2.1 Plateau Selection for Trapezoid Trials

**Wichtig für DR-Analyse:** Bei Trapezoid-Aufgaben sollte die Discharge Rate nur aus dem
**Plateau-Bereich** berechnet werden, nicht aus dem gesamten Signal.

**Workflow:**
1. Referenzsignal wird angezeigt
2. **Manuelle Eingabe:** Gib Start- und End-Zeit des Plateaus ein (z.B. "5.2" und "12.8")
3. Bestätigungs-Plot zeigt das markierte Plateau
4. **Automatische Anwendung** auf ALLE anderen Trapezoid files (relative Zeit)

**Modi:**
- `mode='relative'`: Plateau als % der Signaldauer (empfohlen für standardisierte Protokolle)
- `mode='absolute'`: Plateau als absolute Zeitpunkte in Sekunden

**Manual input advantage:** Works with any matplotlib backend, no additional packages required!

In [ ]:
# ============================================================
# Plateau Selection for Trapezoid Trials (Manual Input)
# ============================================================
# Note: Plateau selection functions are defined in helper.py:
# - select_plateau_interactive(emgfile, title) - shows plot, asks for manual input
# - recalculate_dr_plateau(emgfile, plateau_start, plateau_end)
# - apply_plateau_to_all_trapezoids(condition_result, reference_file, ...)
# ============================================================
print("\nPlateau Selection für Trapezoid files")
print("="*60)
print("\n💡 Workflow:")
print("   1. Wähle EINE Trapezoid-Datei als Referenz")
print("   2. Interaktive Auswahl des Plateaus")
print("   3. Automatische Anwendung auf ALLE anderen Trapezoid files")
print()

# Storage for all plateau results
plateau_results_all = {}

# STEP 1: Select ONE reference trapezoid file and select plateau interactively
if condition_result and OPENHDEMG_AVAILABLE:
    # Find first trapezoid file as reference
    reference_file = None
    reference_filename = None

    for cond_name, cond_data in condition_result['conditions'].items():
        for tracking_type, groups in cond_data['tracking_types'].items():
            if 'Trap' not in tracking_type:
                continue
            for group in groups:
                for file_entry in group.get('files', []):
                    reference_filename = file_entry['filename']
                    reference_file = file_entry['data']
                    break
                if reference_file:
                    break
            if reference_file:
                break
        if reference_file:
            break

    if reference_file:
        print(f"\n📂 Referenz-Datei: {os.path.basename(reference_filename)}")
        print(f"   Signal-Dauer: {len(reference_file['REF_SIGNAL']) / reference_file['FSAMP']:.1f}s")

        # Interactive plateau selection
        start_idx, end_idx = select_plateau_interactive(
            reference_file,
            title=f"Referenz-Plateau: {os.path.basename(reference_filename)}"
        )

        if start_idx is not None:
            # STEP 2: Apply to ALL trapezoid files
            print("\n🔄 Wende Plateau auf alle Trapezoid files an...")

            plateau_results_all = apply_plateau_to_all_trapezoids(
                condition_result=condition_result,
                reference_file=reference_file,
                plateau_start_idx=start_idx,
                plateau_end_idx=end_idx,
                mode='relative'  # or 'absolute'
            )

            # Summary
            print(f"\n✅ Plateau-Berechnung abgeschlossen!")
            print(f"   {len(plateau_results_all)} Trapezoid files verarbeitet")

            # Show example results
            print("\n📊 Beispiel DR-Ergebnisse (erste Datei):")
            example_file = list(plateau_results_all.keys())[0]
            example_result = plateau_results_all[example_file]
            print(f"\n   Datei: {os.path.basename(example_file)}")
            print(f"   Plateau: {example_result['start_time']:.2f}s - {example_result['end_time']:.2f}s")
            print(f"   Dauer: {example_result['duration']:.2f}s")
            print("\n   DR Statistics:")
            print(example_result['dr_results'][['mu_idx', 'mean_dr', 'peak_dr', 'cov_isi']].head().to_string(index=False))
        else:
            print("\n⚠ Plateau Selection cancelled or failed")
    else:
        print("\n⚠ No trapezoid files found")
else:
    print("\n⚠ No conditions available or openhdemg not installed")

#### 5.2.2 Plateau DR - Visualisierung und Export

Verwendung der Plateau-basierten DR-Ergebnisse für weitere Analysen.

In [ ]:
# ============================================================
# Visualize and Export Plateau DR Results
# ============================================================

if plateau_results_all:
    print(f"\n{'='*80}")
    print(f"  Plateau DR - Zusammenfassung")
    print(f"{'='*80}")

    # Collect all DR results into a single DataFrame
    plateau_dr_list = []

    for filename, result in plateau_results_all.items():
        dr_df = result['dr_results'].copy()
        dr_df['filename'] = os.path.basename(filename)
        dr_df['condition'] = result['condition']
        dr_df['tracking_type'] = result['tracking_type']
        dr_df['plateau_start'] = result['start_time']
        dr_df['plateau_end'] = result['end_time']
        dr_df['plateau_duration'] = result['duration']
        plateau_dr_list.append(dr_df)

    df_plateau = pd.concat(plateau_dr_list, ignore_index=True)

    print(f"\n📊 Plateau DR Daten:")
    print(f"   Total MUs: {len(df_plateau)}")
    print(f"   Conditions: {df_plateau['condition'].nunique()}")
    print(f"   Files: {df_plateau['filename'].nunique()}")

    # Summary statistics by condition
    print(f"\n📈 Mean DR by Condition (Plateau only):")
    summary = df_plateau.groupby('condition')['mean_dr'].agg(['count', 'mean', 'std'])
    print(summary.to_string())

    # Visualization: DR comparison across conditions
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Mean DR by condition
    ax = axes[0]
    conditions = sorted(df_plateau['condition'].unique())
    for cond in conditions:
        data = df_plateau[df_plateau['condition'] == cond]['mean_dr'].dropna()
        ax.boxplot([data], positions=[conditions.index(cond)], widths=0.5,
                   patch_artist=True, showmeans=True)
    ax.set_xticks(range(len(conditions)))
    ax.set_xticklabels(conditions, rotation=45, ha='right')
    ax.set_ylabel('Mean DR (pps)', fontsize=12)
    ax.set_title('Plateau Mean DR by Condition', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    # Peak DR by condition
    ax = axes[1]
    for cond in conditions:
        data = df_plateau[df_plateau['condition'] == cond]['peak_dr'].dropna()
        ax.boxplot([data], positions=[conditions.index(cond)], widths=0.5,
                   patch_artist=True, showmeans=True)
    ax.set_xticks(range(len(conditions)))
    ax.set_xticklabels(conditions, rotation=45, ha='right')
    ax.set_ylabel('Peak DR (pps)', fontsize=12)
    ax.set_title('Plateau Peak DR by Condition', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    save_path = save_plot_to_analysis(fig, 'fig_plateau_dr_comparison.png', 'plateau')
    print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

    # Plateau data will be included in consolidated CSV export later
    print(f"✓ Plateau analysis complete: {len(df_plateau)} trapezoid MUs")

else:
    print("\n⚠ No plateau results available.")
    print("   Run the previous cell, to define plateaus.")

### 5.3 Paired Statistical Analysis (Tracked MUs)

Statistical analysis using only **tracked (matched) motor units** from Pre to Post conditions.
Paired tests exploit the within-subject design for higher statistical power.

**Tests used:**
- **Paired t-test** (parametric) - assumes normally distributed differences
- **Wilcoxon signed-rank test** (non-parametric) - no distributional assumptions
- **Cohen's d (paired)** - effect size: |d| < 0.2 small, 0.5 medium, 0.8 large

In [ ]:
# ============================================================
# Paired Statistical Tests: Tracked Motor Units
# ============================================================
if len(df_paired) > 0:
    metrics = ['mean_dr', 'peak_dr', 'rt_pct', 'drt_pct', 'dr_at_rec', 'dr_at_derec', 'cov_isi']
    metric_labels = {
        'mean_dr': 'Mean DR (pps)',
        'peak_dr': 'Peak DR (pps)',
        'rt_pct': 'RT (%REF)',
        'drt_pct': 'DRT (%REF)',
        'dr_at_rec': 'DR@Rec (pps)',
        'dr_at_derec': 'DR@Derec (pps)',
        'cov_isi': 'CoV ISI (%)',
    }

    stat_records = []

    print(f"{'='*100}")
    print(f"  Paired Statistical Analysis (Tracked Motor Units)")
    print(f"{'='*100}")

    for tracking in sorted(df_paired['tracking_type'].unique()):
        for muscle in sorted(df_paired['muscle'].unique()):
            for comp in ['Pre_vs_PostCON', 'Pre_vs_PostEXZ']:
                subset = df_paired[
                    (df_paired['tracking_type'] == tracking) &
                    (df_paired['muscle'] == muscle) &
                    (df_paired['comparison'] == comp)
                ]
                if len(subset) < 3:
                    continue

                intervention = comp.replace('Pre_vs_Post', '')
                print(f"\n  {tracking} | {muscle} | {intervention} (n={len(subset)} pairs)")
                print(f"  {'Metric':<18s} {'Pre':>10s} {'Post':>10s} "
                      f"{'Delta':>10s} {'t':>7s} {'p(t)':>8s} "
                      f"{'d':>7s} {'p(W)':>8s}")
                print(f"  {'-'*18} {'-'*10} {'-'*10} "
                      f"{'-'*10} {'-'*7} {'-'*8} "
                      f"{'-'*7} {'-'*8}")

                for metric in metrics:
                    pre_vals = subset[f'pre_{metric}'].dropna()
                    post_vals = subset[f'post_{metric}'].dropna()
                    delta_vals = subset[f'delta_{metric}'].dropna()

                    if len(delta_vals) < 3:
                        continue

                    # Paired t-test
                    n = len(delta_vals)
                    pre_matched = pre_vals.iloc[:n]
                    post_matched = post_vals.iloc[:n]
                    t_stat, p_ttest = stats.ttest_rel(pre_matched, post_matched)

                    # Wilcoxon signed-rank test
                    try:
                        _, p_wilcox = stats.wilcoxon(delta_vals)
                    except ValueError:
                        p_wilcox = np.nan

                    # Cohen's d (paired): mean_delta / sd_delta
                    cohens_d = delta_vals.mean() / delta_vals.std() if delta_vals.std() > 0 else 0.0

                    sig = '**' if p_ttest < 0.01 else ('*' if p_ttest < 0.05 else ' ')

                    print(f"  {metric_labels[metric]:<18s} "
                          f"{pre_matched.mean():>10.2f} {post_matched.mean():>10.2f} "
                          f"{delta_vals.mean():>+10.2f} {t_stat:>7.2f} {p_ttest:>7.4f}{sig} "
                          f"{cohens_d:>+6.2f} {p_wilcox:>8.4f}")

                    stat_records.append({
                        'tracking_type': tracking,
                        'muscle': muscle,
                        'comparison': comp,
                        'metric': metric,
                        'n_pairs': n,
                        'pre_mean': pre_matched.mean(),
                        'post_mean': post_matched.mean(),
                        'delta_mean': delta_vals.mean(),
                        'delta_sd': delta_vals.std(),
                        't_stat': t_stat,
                        'p_ttest': p_ttest,
                        'cohens_d': cohens_d,
                        'p_wilcoxon': p_wilcox,
                    })

    print(f"\n  * p < 0.05, ** p < 0.01")

    # Save statistical results
    df_stats = pd.DataFrame(stat_records)
    if len(df_stats) > 0:
        stats_path = save_plot_to_analysis(None, 'statistical_tests.csv', 'data')
        df_stats.to_csv(stats_path, index=False)
        print(f"\n✓ Exported: {stats_path.relative_to(WORKFOLDER)}")
else:
    print("No paired data available for statistical analysis")

In [ ]:
# ============================================================
# EXPLORATORY: Spaghetti Plots - Individual MU Changes (Tracked Pairs)
# ============================================================
# Uncomment to generate this exploratory plot (set to True):
if False and len(df_paired) > 0:
    key_metrics = ['mean_dr', 'rt_pct', 'peak_dr', 'cov_isi']
    metric_labels = {
        'mean_dr': 'Mean DR (pps)',
        'rt_pct': 'RT (%REF)',
        'peak_dr': 'Peak DR (pps)',
        'cov_isi': 'CoV ISI (%)',
    }

    for tracking in sorted(df_paired['tracking_type'].unique()):
        t_data = df_paired[df_paired['tracking_type'] == tracking]
        if t_data.empty:
            continue

        muscles = sorted(t_data['muscle'].unique())
        fig, axes = plt.subplots(len(muscles), len(key_metrics),
                                 figsize=(4 * len(key_metrics), 5 * len(muscles)),
                                 squeeze=False)

        for row, muscle in enumerate(muscles):
            m_data = t_data[t_data['muscle'] == muscle]

            for col, metric in enumerate(key_metrics):
                ax = axes[row, col]

                for comp, color, label in [
                    ('Pre_vs_PostCON', '#2563eb', 'CON'),
                    ('Pre_vs_PostEXZ', '#dc2626', 'EXZ'),
                ]:
                    c_data = m_data[m_data['comparison'] == comp]
                    if c_data.empty:
                        continue

                    pre_vals = c_data[f'pre_{metric}'].values
                    post_vals = c_data[f'post_{metric}'].values

                    # Individual MU lines (thin, transparent)
                    for i in range(len(pre_vals)):
                        ax.plot([0, 1], [pre_vals[i], post_vals[i]],
                                color=color, alpha=0.25, linewidth=0.8)

                    # Group mean +/- SEM (bold)
                    pre_mean, post_mean = np.mean(pre_vals), np.mean(post_vals)
                    pre_sem = np.std(pre_vals) / np.sqrt(len(pre_vals))
                    post_sem = np.std(post_vals) / np.sqrt(len(post_vals))

                    offset = -0.05 if comp == 'Pre_vs_PostCON' else 0.05
                    ax.errorbar([0 + offset, 1 + offset], [pre_mean, post_mean],
                                yerr=[pre_sem, post_sem],
                                fmt='-o', color=color, linewidth=2.5, markersize=8,
                                capsize=5, capthick=2, label=f'{label} (n={len(c_data)})',
                                zorder=5)

                ax.set_xticks([0, 1])
                ax.set_xticklabels(['Pre', 'Post'])
                if row == 0:
                    ax.set_title(metric_labels[metric], fontsize=11, fontweight='bold')
                muscle_short = muscle.replace('Right', '').replace('Left', '')
                if col == 0:
                    ax.set_ylabel(f'{muscle_short}\n{metric_labels[metric]}', fontsize=10)
                else:
                    ax.set_ylabel(metric_labels[metric])
                ax.legend(fontsize=8)
                ax.grid(alpha=0.3)

        fig.suptitle(f'{tracking} - Tracked MU Changes: Individual + Mean\u00b1SEM',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        save_path = save_plot_to_analysis(fig, f'fig_paired_spaghetti_{tracking.lower()}.png', 'exploratory')
        print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

In [ ]:
# ============================================================
# EXPLORATORY: Paired Delta CON vs EXZ (Mean +/- SEM, with p-values)
# ============================================================
# Uncomment to generate this exploratory plot (set to True):
if False and len(df_paired) > 0:
    key_metrics = ['mean_dr', 'peak_dr', 'rt_pct', 'cov_isi']
    metric_labels = {
        'mean_dr': '\u0394 Mean DR (pps)',
        'peak_dr': '\u0394 Peak DR (pps)',
        'rt_pct': '\u0394 RT (%REF)',
        'cov_isi': '\u0394 CoV ISI (%)',
    }

    for tracking in sorted(df_paired['tracking_type'].unique()):
        t_data = df_paired[df_paired['tracking_type'] == tracking]
        if t_data.empty:
            continue

        fig, axes = plt.subplots(1, len(key_metrics), figsize=(4 * len(key_metrics), 5))
        if len(key_metrics) == 1:
            axes = [axes]

        for ax, metric in zip(axes, key_metrics):
            x_pos = 0
            xticks, xlabels = [], []

            for muscle in sorted(t_data['muscle'].unique()):
                m_data = t_data[t_data['muscle'] == muscle]
                muscle_short = muscle.replace('Right', '').replace('Left', '')

                for comp, color, label in [
                    ('Pre_vs_PostCON', '#2563eb', 'CON'),
                    ('Pre_vs_PostEXZ', '#dc2626', 'EXZ'),
                ]:
                    deltas = m_data[m_data['comparison'] == comp][f'delta_{metric}'].dropna()
                    if len(deltas) == 0:
                        continue

                    mean_val = deltas.mean()
                    sem_val = deltas.sem()

                    ax.bar(x_pos, mean_val, yerr=sem_val, color=color, alpha=0.7,
                           edgecolor='black', linewidth=0.5, capsize=4, width=0.7)

                    # Add individual data points
                    jitter = np.random.normal(0, 0.08, len(deltas))
                    ax.scatter(np.full(len(deltas), x_pos) + jitter, deltas.values,
                              color=color, alpha=0.4, s=15, edgecolors='none', zorder=3)

                    # Add p-value annotation
                    pre_vals = m_data[m_data['comparison'] == comp][f'pre_{metric}'].dropna()
                    post_vals = m_data[m_data['comparison'] == comp][f'post_{metric}'].dropna()
                    if len(pre_vals) >= 3:
                        n = min(len(pre_vals), len(post_vals))
                        _, p = stats.ttest_rel(pre_vals.iloc[:n], post_vals.iloc[:n])
                        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
                        y_offset = sem_val + abs(mean_val) * 0.1 + 0.2
                        y_pos = mean_val + y_offset if mean_val >= 0 else mean_val - y_offset
                        ax.text(x_pos, y_pos, sig, ha='center', va='bottom' if mean_val >= 0 else 'top',
                                fontsize=9, fontweight='bold')

                    xticks.append(x_pos)
                    xlabels.append(f'{muscle_short}\n{label} (n={len(deltas)})')
                    x_pos += 1
                x_pos += 0.5  # gap between muscles

            ax.set_xticks(xticks)
            ax.set_xticklabels(xlabels, fontsize=8)
            ax.axhline(0, color='black', linewidth=0.8)
            ax.set_ylabel(metric_labels[metric], fontsize=10)
            ax.grid(axis='y', alpha=0.3)

        fig.suptitle(f'{tracking} - Paired \u0394 CON vs EXZ (Mean\u00b1SEM, Tracked MUs)',
                     fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()
        save_path = save_plot_to_analysis(fig, f'fig_paired_delta_{tracking.lower()}.png', 'exploratory')
        print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

    # Paired tracking data will be included in consolidated CSV export later
    print(f"\n✓ Paired tracking analysis complete: {len(df_paired)} tracked MUs")

#### 5.3.1 Washout Verification

**Prüfung der Washout-Wirksamkeit:** Vergleich der MU-Eigenschaften zwischen
**Pre_Intervention** (Block2, vor Training 1) und **Post_Washout** (Block4, vor Training 2).

Wenn der Washout erfolgreich war, sollten beide Messzeitpunkte ähnliche Werte aufweisen.
Signifikante Unterschiede deuten auf unvollständige Erholung hin.

In [ ]:
# ============================================================
# Washout Verification: Pre_Intervention vs Post_Washout
# ============================================================

if len(df_mu) > 0 and 'Pre_Intervention' in df_mu['condition'].values and 'Post_Washout' in df_mu['condition'].values:

    pre_int = df_mu[df_mu['condition'] == 'Pre_Intervention']
    post_wash = df_mu[df_mu['condition'] == 'Post_Washout']

    print(f"{'='*80}")
    print(f"  WASHOUT VERIFICATION")
    print(f"{'='*80}")
    print(f"\nPre_Intervention (Block2): n={len(pre_int)} MUs")
    print(f"Post_Washout (Block4):     n={len(post_wash)} MUs")

    # Statistical comparison
    metrics = ['mean_dr', 'peak_dr', 'rt_pct', 'drt_pct', 'cov_isi']
    metric_labels = {
        'mean_dr': 'Mean DR (pps)',
        'peak_dr': 'Peak DR (pps)',
        'rt_pct': 'RT (%REF)',
        'drt_pct': 'DRT (%REF)',
        'cov_isi': 'CoV ISI (%)',
    }

    print(f"\n{'Metric':<18s} {'Pre_Int':>12s} {'Post_Wash':>12s} {'Diff':>10s} {'t':>7s} {'p':>8s}")
    print(f"{'-'*18} {'-'*12} {'-'*12} {'-'*10} {'-'*7} {'-'*8}")

    for metric in metrics:
        pre_vals = pre_int[metric].dropna()
        post_vals = post_wash[metric].dropna()

        if len(pre_vals) > 0 and len(post_vals) > 0:
            pre_mean = pre_vals.mean()
            post_mean = post_vals.mean()
            diff = post_mean - pre_mean

            # Unpaired t-test (different MUs at each timepoint)
            t_stat, p_val = stats.ttest_ind(pre_vals, post_vals)

            sig = '**' if p_val < 0.01 else ('*' if p_val < 0.05 else '')

            print(f"{metric_labels[metric]:<18s} {pre_mean:>12.2f} {post_mean:>12.2f} "
                  f"{diff:>+10.2f} {t_stat:>7.2f} {p_val:>7.4f}{sig}")

    print(f"\n* p < 0.05, ** p < 0.01")
    print(f"\n{'='*80}")

    # Visualization
    fig, axes = plt.subplots(1, len(metrics), figsize=(4*len(metrics), 5))
    if len(metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        data_to_plot = []
        labels = []

        for tracking in sorted(df_mu['tracking_type'].unique()):
            for muscle in sorted(df_mu['muscle'].unique()):
                muscle_short = muscle.replace('Right', '').replace('Left', '')

                pre_data = df_mu[(df_mu['condition'] == 'Pre_Intervention') &
                                 (df_mu['tracking_type'] == tracking) &
                                 (df_mu['muscle'] == muscle)][metric].dropna()

                post_data = df_mu[(df_mu['condition'] == 'Post_Washout') &
                                  (df_mu['tracking_type'] == tracking) &
                                  (df_mu['muscle'] == muscle)][metric].dropna()

                if len(pre_data) > 0:
                    data_to_plot.append(pre_data.values)
                    labels.append(f'{muscle_short}\n{tracking[:4]}\nPre')

                if len(post_data) > 0:
                    data_to_plot.append(post_data.values)
                    labels.append(f'{muscle_short}\n{tracking[:4]}\nWash')

        if data_to_plot:
            bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
            for i, patch in enumerate(bp['boxes']):
                color = '#3b82f6' if 'Pre' in labels[i] else '#10b981'
                patch.set_facecolor(color)
                patch.set_alpha(0.6)

            ax.set_ylabel(metric_labels[metric])
            ax.set_title(metric_labels[metric], fontsize=11, fontweight='bold')
            ax.tick_params(axis='x', labelsize=8, rotation=45)
            ax.grid(axis='y', alpha=0.3)

    fig.suptitle('Washout Verification: Pre_Intervention vs Post_Washout',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    save_path = save_plot_to_analysis(fig, 'fig_washout_verification.png', 'washout')
    print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

else:
    print("Washout verification requires Pre_Intervention and Post_Washout data")

## 6. PIC Analysis (Persistent Inward Currents)

Analyse von **Persistent Inward Currents (PICs)** aus **Pyramid**-Kontraktionen.

PICs können die neuronale Erregbarkeit verstärken und sind wichtig für die Aufrechterhaltung
der motorischen Aktivität. Sie werden durch paired motor unit analysis geschätzt.

**Methoden:**
- **Delta F (ΔF)**: Änderung der Control-MU Firing Rate zwischen Test-MU Rekrutierung und Derekrutierung
- Basiert auf Gorassini et al. (2002) und Skarabot et al. (2023)

**Voraussetzungen:**
- Openhdemg library mit PIC-Modul installiert
- Pyramid-Aufnahmen (langsame Rampen-Kontraktionen)

In [ ]:
# ============================================================
# SVR Fitting: Smooth Discharge Rate Estimates
# ============================================================
# Support Vector Regression (SVR) liefert geglättete DR-Schätzungen,
# die für Delta F-Berechnung benötigt werden.

if not OPENHDEMG_AVAILABLE:
    print("⚠ openhdemg not available - PIC analysis requires openhdemg")
else:
    from openhdemg.library import tools as emg_tools
    from openhdemg.library import pic as emg_pic

    # Select pyramid files for PIC analysis
    pyramid_files = []

    if condition_result:
        for cond_name, cond_data in condition_result['conditions'].items():
            for tracking_type, groups in cond_data['tracking_types'].items():
                if 'Pyramid' not in tracking_type:
                    continue
                for group in groups:
                    emgfile = group.get('concatenated')
                    if emgfile and emgfile['NUMBER_OF_MUS'] >= 2:
                        pyramid_files.append({
                            'condition': cond_name,
                            'muscle': group.get('muscle', 'Unknown'),
                            'emgfile': emgfile,
                            'name': group.get('name', 'Unknown'),
                        })

    print(f"Found {len(pyramid_files)} pyramid files with >= 2 MUs")

    # Compute SVR fits for each pyramid file
    svr_results = []

    for pf in pyramid_files:
        emgfile = pf['emgfile']
        print(f"\nSVR fitting: {pf['condition']} | {pf['muscle']}")
        print(f"  MUs: {emgfile['NUMBER_OF_MUS']}")

        try:
            # Sort MUs by recruitment threshold
            emgfile_sorted = emg_tools.sort_mus(emgfile=emgfile)

            # Compute SVR smooth fits
            svrfits = emg_tools.compute_svr(
                emgfile_sorted,
                gammain=1/1.6,
                regparam=1/0.370,
                endpointweights_numpulses=5,
                endpointweights_magnitude=5,
                discontfiring_dur=1.0,
            )

            svr_results.append({
                'condition': pf['condition'],
                'muscle': pf['muscle'],
                'name': pf['name'],
                'emgfile': emgfile_sorted,
                'svrfits': svrfits,
            })

            print(f"  ✓ SVR fitting successful")

        except Exception as e:
            print(f"  ✗ SVR fitting failed: {e}")

    print(f"\n✓ SVR fitting complete: {len(svr_results)} files processed")

In [ ]:
# ============================================================
# Delta F Computation
# ============================================================

deltaf_results = []

if OPENHDEMG_AVAILABLE and len(svr_results) > 0:
    for svr_res in svr_results:
        emgfile = svr_res['emgfile']
        smoothfits = svr_res['svrfits']['gensvr']

        print(f"\nDelta F: {svr_res['condition']} | {svr_res['muscle']}")

        try:
            # Compute delta F (test unit average method)
            delta_f = emg_pic.compute_deltaf(
                emgfile=emgfile,
                smoothfits=smoothfits,
                average_method="test_unit_average",
                normalisation="False",
                recruitment_difference_cutoff=1.0,
                corr_cutoff=0.7,
                controlunitmodulation_cutoff=0.5,
                clean=True,
            )

            # Also compute all MU pairs
            delta_f_all = emg_pic.compute_deltaf(
                emgfile=emgfile,
                smoothfits=smoothfits,
                average_method="all",
                normalisation="False",
                recruitment_difference_cutoff=1.0,
                corr_cutoff=0.7,
                controlunitmodulation_cutoff=0.5,
                clean=True,
            )

            deltaf_results.append({
                'condition': svr_res['condition'],
                'muscle': svr_res['muscle'],
                'name': svr_res['name'],
                'delta_f': delta_f,
                'delta_f_all': delta_f_all,
                'emgfile': emgfile,
            })

            # Display results
            valid_df = delta_f.dropna()
            print(f"  Valid delta F: {len(valid_df)}/{len(delta_f)} MUs")
            if len(valid_df) > 0:
                print(f"  Mean ΔF: {valid_df['dF'].mean():.3f} Hz")
                print(f"  Range: {valid_df['dF'].min():.3f} - {valid_df['dF'].max():.3f} Hz")
                print()
                print(valid_df.to_string(index=False))

        except Exception as e:
            print(f"  ✗ Delta F computation failed: {e}")

    print(f"\n✓ Delta F complete: {len(deltaf_results)} files processed")
else:
    print("No SVR results available for Delta F computation")

In [ ]:
# ============================================================
# Figure 9: Delta F Results Visualization
# ============================================================

if len(deltaf_results) > 0:
    # Collect all delta F values
    all_df_records = []

    for df_res in deltaf_results:
        delta_f = df_res['delta_f']
        for _, row in delta_f.iterrows():
            if not np.isnan(row['dF']):
                all_df_records.append({
                    'condition': df_res['condition'],
                    'muscle': df_res['muscle'],
                    'MU': row['MU'],
                    'dF': row['dF'],
                })

    df_deltaf = pd.DataFrame(all_df_records)

    if len(df_deltaf) > 0:
        print(f"\nDelta F dataset: {len(df_deltaf)} valid MUs")

        # Figure: Delta F by condition and muscle
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Boxplot by condition
        ax = axes[0]
        conditions = sorted(df_deltaf['condition'].unique())
        data_by_cond = [df_deltaf[df_deltaf['condition'] == c]['dF'].values
                        for c in conditions]
        bp = ax.boxplot(data_by_cond, labels=conditions, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#9333ea')
            patch.set_alpha(0.6)
        ax.set_ylabel('Delta F (Hz)', fontsize=12)
        ax.set_title('Delta F by Condition', fontsize=13, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)
        ax.tick_params(axis='x', rotation=45)

        # Boxplot by muscle
        ax = axes[1]
        muscles = sorted(df_deltaf['muscle'].unique())
        data_by_muscle = [df_deltaf[df_deltaf['muscle'] == m]['dF'].values
                          for m in muscles]
        muscle_labels = [m.replace('Right', '').replace('Left', '') for m in muscles]
        bp = ax.boxplot(data_by_muscle, labels=muscle_labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#9333ea')
            patch.set_alpha(0.6)
        ax.set_ylabel('Delta F (Hz)', fontsize=12)
        ax.set_title('Delta F by Muscle', fontsize=13, fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        plt.tight_layout()
        save_path = save_plot_to_analysis(fig, 'fig_deltaf_pyramids.png', 'pic')
        print(f"✓ Saved: {save_path.relative_to(WORKFOLDER)}")

        # Delta F data will be included in consolidated CSV export later
        print(f"\n✓ PIC (Delta F) analysis complete: {len(df_deltaf)} comparisons")

        # Summary statistics
        print(f"\n{'='*60}")
        print("Delta F Summary Statistics")
        print(f"{'='*60}")
        summary = df_deltaf.groupby(['condition', 'muscle'])['dF'].agg(['count', 'mean', 'std', 'min', 'max'])
        print(summary.round(3))

    else:
        print("No valid delta F values to visualize")
else:
    print("No delta F results available")

## 6.5. Consolidated Data Export

Export all motor unit data to a single CSV file for cross-patient statistical analysis.

In [ ]:
# ============================================================
# Consolidated CSV Export
# ============================================================

df_parts = []

# 1. Basic MU properties (if available)
if 'df_properties' in locals() and df_properties is not None:
    df_parts.append(df_properties)

# 2. Properties by condition
if 'df_mu' in locals() and df_mu is not None:
    df_parts.append(df_mu)

# 3. Paired tracking data (if available)
if 'df_paired' in locals() and df_paired is not None:
    # Add tracking flag
    df_paired_export = df_paired.copy()
    df_paired_export['Is_Tracked'] = True
    df_parts.append(df_paired_export)

# 4. Plateau analysis (trapezoids, if available)
if 'df_plateau' in locals() and df_plateau is not None:
    df_parts.append(df_plateau)

# 5. Delta F (PIC analysis, pyramids, if available)
if 'df_deltaf' in locals() and df_deltaf is not None:
    df_parts.append(df_deltaf)

# Merge all per-MU data
if df_parts:
    # Concatenate with outer join to keep all columns
    df_complete = pd.concat(df_parts, ignore_index=True, sort=False)

    # Remove exact duplicates (prefer later entries with more complete data)
    df_complete = df_complete.drop_duplicates(
        subset=['File', 'MU_Index'] if 'File' in df_complete.columns and 'MU_Index' in df_complete.columns else None,
        keep='last'
    )

    # Export consolidated CSV
    output_path, row_count = export_consolidated_csv(
        [df_complete],
        output_name='motor_unit_analysis_complete.csv'
    )

    print(f"\n{'='*60}")
    print(f"CONSOLIDATED EXPORT")
    print(f"{'='*60}")
    print(f"  File: {output_path.relative_to(WORKFOLDER)}")
    print(f"  Rows: {row_count} motor units")
    print(f"  Columns: {len(df_complete.columns)}")

    # Summary by key dimensions
    if 'Condition' in df_complete.columns:
        print(f"  Conditions: {df_complete['Condition'].nunique()}")
    if 'Muscle' in df_complete.columns:
        print(f"  Muscles: {df_complete['Muscle'].nunique()}")
    if 'tracking_type' in df_complete.columns:
        print(f"  Tracking types: {df_complete['tracking_type'].nunique()}")

    print(f"{'='*60}")
else:
    print("⚠ No data available for consolidated export")

## 7. Custom Analysis Section

**Template for your own analysis**

Use this section to add your custom analysis code. Examples:
- Cross-correlations between motor units
- Coherence analysis with reference signal
- Custom recruitment pattern analysis
- Export to custom formats (Excel, MATLAB, etc.)

In [ ]:
# Your custom analysis code here

# Example: Export to Excel
if len(emgfiles) > 0:
    # Create Excel workbook with multiple sheets
    output_excel = WORKFOLDER / "custom_analysis.xlsx"

    # Example: Export discharge times to Excel
    # with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    #     for emg_entry in emgfiles:
    #         # Your export logic
    #         pass

    print("Custom analysis placeholder - add your code here")

## 7.5. Analysis Summary

Overview of all generated files and results.

In [ ]:
# ============================================================
# Analysis Summary
# ============================================================

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)

# List all generated files
analysis_dir = Path(WORKFOLDER) / 'analysis'

if analysis_dir.exists():
    print("\nGenerated files:")

    for subfolder in ['comparisons', 'timelines', 'tracking', 'plateau', 'washout', 'pic', 'data', 'exploratory']:
        folder_path = analysis_dir / subfolder
        if folder_path.exists():
            files = list(folder_path.glob('*'))
            if files:
                print(f"\n  {subfolder}/")
                for f in sorted(files):
                    size_kb = f.stat().st_size / 1024
                    print(f"    - {f.name} ({size_kb:.1f} KB)")

    print("\n" + "="*60)
    print(f"All results saved to: {analysis_dir}")
    print("="*60)
else:
    print("⚠ Analysis directory not found")

## 8. Export Results

Save your analysis results to files for publication or further processing.

In [ ]:
# Export analysis results

# Example exports:
# 1. Motor unit properties DataFrame (already saved above)
# 2. Custom figures
# 3. Summary statistics

print("=" * 60)
print("Export Summary")
print("=" * 60)
print(f"Workfolder: {WORKFOLDER}")
print(f"Files available:")
print(f"  - motor_unit_properties.csv (MU statistics)")
print(f"  - [Add your custom exports here]")
print("\n✓ Analysis complete!")